# Cluster the landscape — k-means

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/main/notebooks/06_kmeans_clustering.ipynb)

So far every map in this series showed a **number** — an index value, a
temperature. This notebook and the next one produce **categories**: which kind of
surface is each pixel? There are two very different routes to that, and we take
them in order:

- here, **unsupervised**: `k-means` is told nothing but *"find k groups"* — and
  you will discover what that silence costs;
- in notebook 07, **supervised**: a random forest learns land-cover classes from
  labeled examples.

Everything runs server-side in Earth Engine on one summer composite — no
rendering pipeline, just the package's auth, AOI handling and cloud masking.

## 1 · Install (Colab only)

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "main"
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code
# pip resolves by version, which doesn't change per commit on a branch --
# --force-reinstall pulls the latest commit. After a code update: Runtime ->
# Restart session, then Run all, so the new module is re-imported.

## 2 · Authenticate

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"

from gee_animation import auth
auth.init(PROJECT)

## 3 · The scene — Grumsin and its surroundings

Classification needs **variety**: a frame holding only beech canopy has nothing to
separate. The Grumsin footprint plus a 3 km collar brings in farmland, lakes and
settlements — enough distinct surfaces for the method to have something to find.

In [ ]:
from pathlib import Path
import ee
from gee_animation import aoi

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "aoi").is_dir()), None)
if REPO is not None:
    REGION = str(REPO / "docs" / "aoi" / "wne" / "wne.geojson")
else:   # Colab: fetch the same file
    import urllib.request
    RAW = ("https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/main"
           "/docs/aoi/wne/wne.geojson")
    urllib.request.urlretrieve(RAW, "wne.geojson")
    REGION = "wne.geojson"

START, END = "2025-06-01", "2025-09-01"     # one summer, end exclusive

region_geom = aoi.parse({"geojson": REGION})
frame_bbox = list(aoi.frame_bbox_from_region(region_geom, 3000.0))
frame_geom = aoi.parse({"bbox": frame_bbox})
print("frame bbox:", [round(v, 4) for v in frame_bbox], "(footprint + 3 km)")

## 4 · One cloud-free summer composite

The animation pipeline keeps only its computed index band, so here we assemble the
multiband input ourselves — from the same three pieces of the package's `Sensor`
object that `collection.build` uses: the source collection, the per-pixel cloud
mask (SCL + s2cloudless, which first needs its probability band joined on), and
the six **harmonised reflectance aliases** from notebook 05 (`blue … swir2`).
The summer median of that is the one image everything below works on.

In [ ]:
from gee_animation import products

sensor = products.SENSORS["sentinel2"]
BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]

coll = (sensor.collection(ee)
        .filterDate(START, END)
        .filterBounds(frame_geom)
        .filter(ee.Filter.lte(sensor.scene_cloud_property, 70)))   # scene-level pre-filter
coll = sensor.attach_aux(coll, START, END, frame_geom, ee_module=ee)  # join s2cloudless probabilities
coll = (coll.map(lambda img: sensor.mask_clouds(img, ee))     # per-pixel: SCL classes + cloud_prob
            .map(lambda img: sensor.reflectance(img, ee)))    # six harmonised bands, reflectance 0-1
composite = coll.median().clip(frame_geom)

print(coll.size().getInfo(), "cloud-masked scenes in the composite")

In [ ]:
from IPython.display import Image, display

def thumb(image, vis, title):
    url = image.getThumbURL({**vis, "region": frame_geom, "dimensions": 768})
    print(title)
    display(Image(url=url))

thumb(composite, {"bands": ["red", "green", "blue"], "min": 0, "max": 0.17},
      "True colour, summer median " + START[:4])

## 5 · k-means — let the data group itself

`ee.Clusterer.wekaKMeans(k)` looks at a sample of pixels — six reflectance values
each — and places k centres so that every pixel is near one of them. No labels,
no training polygons, no opinion about what a forest is. The output is a map of
the numbers 0 … k−1.

In [ ]:
from IPython.display import HTML

K = 6
PALETTE6 = ["1f78b4", "33a02c", "b2df8a", "fdbf6f", "e31a1c", "cab2d6"]

def legend(entries):
    """entries: [(hex_color, label)] -> one row of colour swatches."""
    row = "".join(
        f'<span style="display:inline-block;margin-right:14px">'
        f'<span style="display:inline-block;width:14px;height:14px;background:#{c};'
        f'border:1px solid #666;margin-right:4px;vertical-align:-2px"></span>{label}</span>'
        for c, label in entries)
    return HTML(f'<div style="font-family:sans-serif;font-size:13px">{row}</div>')

sample = composite.select(BANDS).sample(region=frame_geom, scale=20,
                                        numPixels=5000, seed=1)
clusterer = ee.Clusterer.wekaKMeans(K).train(sample)
clustered = composite.select(BANDS).cluster(clusterer)

thumb(clustered, {"min": 0, "max": K - 1, "palette": PALETTE6},
      f"k-means, k={K} — the colours are assigned by us, the groups by the data")
display(legend([(c, f"cluster {k}") for k, c in enumerate(PALETTE6)]))

## 6 · Naming the numbers

The map looks like a land-cover map, but the legend is `0 1 2 3 4 5`. To attach
meaning you have to interrogate each cluster's spectrum yourself — mean NDVI
separates vegetation from everything else, and `swir1` near zero is the signature
of water:

In [ ]:
named = clustered.addBands(composite).sample(region=frame_geom, scale=20,
                                             numPixels=2000, seed=2).getInfo()
rows = [f["properties"] for f in named["features"]]
CLUSTER_STATS = {}
print(f"{'cluster':>7} {'n':>5} {'ndvi':>6} {'swir1':>6}   hint")
for k in range(K):
    rs = [r for r in rows if r["cluster"] == k]
    if not rs:
        continue
    nir = sum(r["nir"] for r in rs) / len(rs); red = sum(r["red"] for r in rs) / len(rs)
    ndvi = (nir - red) / (nir + red + 1e-9)
    swir1 = sum(r["swir1"] for r in rs) / len(rs)
    hint = "water?" if swir1 < 0.03 else "vegetation?" if ndvi > 0.6 else "…"
    CLUSTER_STATS[k] = {"n": len(rs), "ndvi": round(ndvi, 2), "swir1": round(swir1, 3)}
    print(f"{k:>7} {len(rs):>5} {ndvi:>6.2f} {swir1:>6.3f}   {hint}")

## 7 · Write the names down — `cluster_names.json`

The naming decision deserves to outlive this kernel. The next cell writes a
**template mapping** with each cluster's colour and spectral hints and an empty
`"name"` — fill those in (locally: any editor; in Colab: the Files sidebar on
the left, double-click the file). Re-running the cell never overwrites a file
that already exists, so your names survive *Run all*.

In [ ]:
import json
from pathlib import Path

NAMES_FILE = Path("cluster_names.json")
if not NAMES_FILE.exists():                 # never clobber names you already wrote
    template = {str(k): {"color": f"#{PALETTE6[k]}", **CLUSTER_STATS.get(k, {}), "name": ""}
                for k in range(K)}
    NAMES_FILE.write_text(json.dumps(template, indent=2) + "\n")
    print(f'wrote {NAMES_FILE} — fill in the empty "name" fields, then run the next cell')
else:
    print(f"{NAMES_FILE} exists — keeping your names")
print(NAMES_FILE.read_text())

In [ ]:
mapping = json.loads(NAMES_FILE.read_text())
names = {int(k): (v["name"] or f"cluster {k} (unnamed)") for k, v in mapping.items()}

thumb(clustered, {"min": 0, "max": K - 1, "palette": PALETTE6},
      "the same map, with your legend")
display(legend([(PALETTE6[k], names[k]) for k in range(K)]))

A cluster with NDVI ≈ 0.9 is *some kind of* vegetation, and one with swir1 ≈ 0.01
is almost certainly water — but whether cluster 3 is "crops" or "grassland" the
data cannot say. Change `K` and all the boundaries redraw. Your
`cluster_names.json` makes the judgement explicit, and that is all it is: your
judgement, written down, checked by nobody. That gap between *groups* and
*meaning* is the whole limitation of unsupervised classification, and
**notebook 07** closes it — by paying the price of needing labels.

## Make it yours

- Swap `REGION` for your own GeoJSON (a bbox works too) — mixed land use makes
  far better clusters than pure forest.
- Turn the `K` knob: at 4 the lakes and fields merge, at 10 the beech stand
  splits by illumination. Watch the names get harder, not easier.
- Cluster on fewer bands (`["nir", "red"]`): how much structure survives?
- Add a texture or elevation band to `BANDS` and see whether the clusters start
  following terrain instead of cover.